# Lab 5 Projesi: Türkçe Kitap Tanıtım Yazılarıyla Naive Bayes
Bu notebook şarkı sözü notebook'unun aynı akışını kullanır: temizle → kelime sayılarına çevir → `MultinomialNB` → F1.

1. **Kategori tahmini:** tanıtım yazısından kitabın kategorisini bul
2. **Yazar tahmini:** tanıtım yazısından kitabın yazarını bul

Veri: Kaggle *Turkish Book Dataset* (`Turkish_Book_Dataset_Kaggle_V2.csv`)

In [ ]:
import numpy as np
import pandas as pd
import string
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.naive_bayes import MultinomialNB

In [ ]:
df = pd.read_csv("Turkish_Book_Dataset_Kaggle_V2.csv", dtype=str)
print(df.shape)
df.head()

## 1. Veri temizliği
Dosyada yaklaşık 120 bin satır tamamen boş. Ayrıca tanıtım yazısı çok kısa olan ve aynı yazının tekrar ettiği kayıtlar var. Bunları eliyoruz.

In [ ]:
print(df.isna().sum())

df_clean = df.dropna(subset=["book_detail", "book_category_name", "book_author"]).copy()
df_clean = df_clean[df_clean.book_detail.str.len() >= 50]
df_clean = df_clean.drop_duplicates("book_detail")
print("Temizlik sonrası kitap sayısı:", len(df_clean))

In [ ]:
df_clean.book_category_name.value_counts()

## 2. Metin temizleme
Şarkı sözü notebook'undaki `clean` fonksiyonunun aynısı, tek fark Türkçe `I/İ` harflerini doğru küçültmek (`I` → `ı`, `İ` → `i`) ve tipografik tırnakları da silmek.

In [ ]:
tablo = str.maketrans("", "", string.punctuation + "…“”‘’")

def clean(text):
    text = text.replace("I", "ı").replace("İ", "i").lower()
    return " ".join(text.translate(tablo).split())

df_clean["clean"] = df_clean.book_detail.apply(clean)
df_clean[["book_category_name", "clean"]].head()

## 3. Kategori tahmini
Hedef: `book_category_name` (5 sınıf). Sınıflar dengesiz olduğu için başarıyı **macro F1** ile de ölçüyoruz.

In [ ]:
X = df_clean.clean.to_numpy()
y = df_clean.book_category_name.to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), len(X_test))

In [ ]:
vectorizer = CountVectorizer(min_df=2)
Xv_train = vectorizer.fit_transform(X_train)
Xv_test = vectorizer.transform(X_test)
len(vectorizer.vocabulary_)

In [ ]:
model = MultinomialNB()
model.fit(Xv_train, y_train)

pred_train = model.predict(Xv_train)
pred_test = model.predict(Xv_test)
print("Train accuracy:", accuracy_score(y_train, pred_train))
print("Test accuracy :", accuracy_score(y_test, pred_test))
print("Train macro F1:", f1_score(y_train, pred_train, average="macro"))
print("Test macro F1 :", f1_score(y_test, pred_test, average="macro"))

In [ ]:
print(classification_report(y_test, pred_test))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.tight_layout()
plt.show()

### Her kategoriyi en çok ayıran kelimeler
`feature_log_prob_` her sınıfta bir kelimenin log olasılığını verir. Bir sınıfın değerinden diğer sınıfların ortalamasını çıkararak o sınıfa özgü kelimeleri buluruz.

In [ ]:
kelimeler = vectorizer.get_feature_names_out()
lp = model.feature_log_prob_
for i, sinif in enumerate(model.classes_):
    diger = np.delete(lp, i, axis=0).mean(axis=0)
    en_iyi = np.argsort(lp[i] - diger)[::-1][:10]
    print(f"{sinif:25s}", ", ".join(kelimeler[en_iyi]))

### Kendi cümlenle dene
Şarkı notebook'undaki `zeki_muren_sozu_gibi` hücresinin karşılığı.

In [ ]:
def kategori_tahmin(metin):
    vec = vectorizer.transform([clean(metin)])
    olasiliklar = model.predict_proba(vec)[0]
    return dict(zip(model.classes_, olasiliklar.round(3)))

kategori_tahmin("Stres, kaygı ve depresyonla başa çıkmak için bilişsel davranışçı terapi teknikleri")

In [ ]:
kategori_tahmin("Aşk, ayrılık ve hüzün dolu, İstanbul'da geçen etkileyici bir roman")

### Deney: `alpha` değeri
Laplace düzeltmesi (`alpha`) kelime olasılığı sıfır olmasın diye eklenir. Farklı değerlerin etkisine bakalım.

In [ ]:
for alpha in [0.01, 0.1, 0.5, 1.0, 2.0]:
    m = MultinomialNB(alpha=alpha).fit(Xv_train, y_train)
    p = m.predict(Xv_test)
    print(f"alpha={alpha:<5} acc={accuracy_score(y_test, p):.4f}  macroF1={f1_score(y_test, p, average='macro'):.4f}")

## 4. Yazar tahmini
Hedef: `book_author`. Kolektif (çok yazarlı) kitapları çıkarıp en çok kitabı olan 10 yazarı alıyoruz.

**Dikkat:** Elimizde yazarın kendi metni yok, kitabın *tanıtım yazısı* var. Model bu yüzden yazarın üslubunu değil, yazarın işlediği konuları ve tanıtım yazılarındaki ortak kalıpları öğrenir.

In [ ]:
yazarlar_df = df_clean[df_clean.book_author != "Kolektif"]
top_yazarlar = yazarlar_df.book_author.value_counts().head(10).index.tolist()
df_yazar = yazarlar_df[yazarlar_df.book_author.isin(top_yazarlar)].copy()
df_yazar.book_author.value_counts()

### Veri sızıntısı kontrolü
İlk denemede en belirleyici kelimeler yazarın kendi adıydı (zweig, freud, tolstoy...). Tanıtım yazısında yazarın adı geçtiği için model üslubu değil adı yakalıyor. Bu yüzden yazarın adının parçalarını (ve `zweigın`, `freudun` gibi ek almış hallerini) metinden siliyoruz.

In [ ]:
def yazar_adini_sil(row):
    parcalar = [p for p in clean(row["book_author"]).split() if len(p) >= 3 and p not in ("prof", "sir")]
    kelimeler = [k for k in row["clean"].split() if not any(k.startswith(p) for p in parcalar)]
    return " ".join(kelimeler)

df_yazar["clean_adsiz"] = df_yazar.apply(yazar_adini_sil, axis=1)
df_yazar[["book_author", "clean_adsiz"]].head()

In [ ]:
Xa = df_yazar.clean_adsiz.to_numpy()
ya = df_yazar.book_author.to_numpy()
Xa_train, Xa_test, ya_train, ya_test = train_test_split(Xa, ya, test_size=0.2, random_state=42, stratify=ya)

vec_a = CountVectorizer(min_df=2)
Xav_train = vec_a.fit_transform(Xa_train)
Xav_test = vec_a.transform(Xa_test)

model_a = MultinomialNB().fit(Xav_train, ya_train)
pa = model_a.predict(Xav_test)
print("Test accuracy:", accuracy_score(ya_test, pa))
print("Test macro F1:", f1_score(ya_test, pa, average="macro"))
print(classification_report(ya_test, pa))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(ya_test, pa, ax=ax, xticks_rotation=75, cmap="Blues", colorbar=False)
plt.tight_layout()
plt.show()

In [ ]:
kelimeler_a = vec_a.get_feature_names_out()
lp_a = model_a.feature_log_prob_
for i, yazar in enumerate(model_a.classes_):
    diger = np.delete(lp_a, i, axis=0).mean(axis=0)
    en_iyi = np.argsort(lp_a[i] - diger)[::-1][:8]
    print(f"{yazar:32s}", ", ".join(kelimeler_a[en_iyi]))

## 5. Sonuç ve yorum
- Kategori tahmini yüksek doğruluk verir; tanıtım yazısındaki kelimeler konuyu açıkça belirler. Küçük sınıflarda (psikoloji, bilim) F1 daha düşüktür, çünkü örnek sayısı azdır ve sınıflar birbirine yakın konular içerir.
- Yazar tahmini, yazar adı silindikten sonra da çalışır, ama bunun nedeni üslup değil; yazarın tekrar eden konuları, kitap adları ve tanıtım kalıplarıdır (ör. dergi serilerindeki "sayımızda"). Gerçek üslup analizi için yazarın kendi metinleri gerekir.
- `alpha` deneyi, Laplace düzeltmesinin sonucu ne kadar etkilediğini gösterir.

**Olası geliştirmeler:** `TfidfVectorizer` karşılaştırması, Türkçe kök bulma (Zemberek), `ComplementNB` ile dengesiz sınıfları ele alma.